# Diagnóstico de marca empleadora — Accenture vs. peers
## Sentiment analysis (transformers) + topic modeling (BERTopic) sobre reseñas de Glassdoor

**Encargo de RRHH:** identificar los puntos fuertes y las áreas de mejora de Accenture como empleador, a partir de lo que dicen sus empleados en Glassdoor, y compararlos con los de sus competidores directos.

**Peers elegidos:** Capgemini, Deloitte, Cognizant Technology Solutions e Infosys. Son empresas de consultoría y tecnología con una base de talento comparable y un perfil de empleado muy próximo al de Accenture en servicios, proyectos digitales y gestión de clientes.

| Sección | Contenido |
|---|---|
| 0 | Carga y preparación |
| 1 | EDA: rating, recomendación, empleados actuales vs. antiguos |
| 2 | Sentimiento en pros y cons (control de calidad) |
| 3 | Topic modeling con BERTopic |
| 4 | Comparación con peers: peso de cada topic y balance neto |
| 5 | Empleados actuales vs. antiguos |
| 6 | Conclusiones y recomendaciones para RRHH |

In [ ]:
# En Colab: transformers ya viene instalado; BERTopic no.
# !pip install -q bertopic sentence-transformers

---
## 0. Carga y preparación

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


# Empresa objetivo y peers del caso práctico
TARGET = 'Accenture'
PEERS = ['Capgemini', 'Deloitte', 'Cognizant Technology Solutions', 'Infosys']
FIRMS = [TARGET] + PEERS

# Tope de reseñas por empresa: limita el tiempo de cómputo y evita que el peer
# más grande domine el modelo de topics conjunto.
MAX_POR_EMPRESA = 1000
SEED = 42

base_path = '../data/raw/glassdoor_reviews_hr.parquet'
raw = pd.read_parquet(base_path, filters=[('firm', 'in', FIRMS)])
print('Reseñas disponibles por empresa:')
print(raw['firm'].value_counts())

df = pd.concat([g.sample(n=min(MAX_POR_EMPRESA, len(g)), random_state=SEED)
                for _, g in raw.groupby('firm')], ignore_index=True)
print(f'\nMuestra de trabajo: {len(df):,} reseñas')

Reseñas disponibles por empresa:
firm
Citi             14609
J-P-Morgan       11100
Barclays          9863
HSBC-Holdings     6404
Deutsche-Bank     5474
Name: count, dtype: int64

Muestra de trabajo: 5,000 reseñas


La columna `current` mezcla situación y antigüedad (`"Former Employee, more than 3 years"`). La separo en dos variables. También traduzco `recommend` (v = sí, o = sin opinión, x = no) y limpio saltos de línea del texto.

In [4]:
def situacion(s):
    s = str(s).lower()
    if s.startswith('current'):
        return 'Actual'
    if s.startswith('former'):
        return 'Antiguo'
    return 'Desconocido'

def antiguedad(s):
    m = re.search(r'(less|more) than (\d+) year', str(s))
    return f'{m.group(1)} than {m.group(2)}y' if m else 'n/d'

df['situacion'] = df['current'].map(situacion)
df['antiguedad'] = df['current'].map(antiguedad)
df['recomienda'] = df['recommend'].map({'v': 'Sí', 'o': 'Neutral', 'x': 'No'})
df['date_review'] = pd.to_datetime(df['date_review'])
df['anio'] = df['date_review'].dt.year

for col in ['pros', 'cons']:
    df[col] = df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

print(df['situacion'].value_counts())
df[['firm', 'date_review', 'situacion', 'antiguedad', 'overall_rating', 'recomienda', 'pros', 'cons']].head()

situacion
Actual     30279
Antiguo    17171
Name: count, dtype: int64


,firm,date_review,situacion,antiguedad,overall_rating,recomienda,pros,cons
0,Barclays,2023-03-02,Antiguo,more than 1y,4.0,Sí,Good work life balance Had a great management ...,I would say.. May be the compensation
1,Barclays,2019-01-23,Actual,more than 1y,4.0,Neutral,"Great people, good atmosphere, work-life balance","Overly bureaucratic, too many committees and f..."
2,Barclays,2021-05-08,Actual,n/d,5.0,Sí,"Good culture, environment and working with sma...",Too much work. Lean team
3,Barclays,2019-05-11,Antiguo,more than 5y,4.0,Sí,Barclays are a really amazing company to work ...,"As mentioned above, the floors of head office ..."
4,Barclays,2021-10-23,Actual,n/d,4.0,Sí,Flexible Lovely team Inclusive Manager support...,Genuinely cannot think of any cons


---
## 1. EDA

- Para cada firma muestra el numero de reseñas, rating medio % que recomiendan, % de empleados actuales
- Compara taring dado por empleados actuales vs empleados antiguos
- Evolucion del rating
- ...

---
## 2. Sentimiento en pros y cons (control de calidad)

Los pros y cons  ya nos darían un sentmiento, haremos un control de calidad para verificar que el sentimiento predom,inande ten pros es el positivo y en cons es en negativo.

Para ellos puedes emplear alguno de los modelos transaformer ya preentrenados vistos en clase o Vader de nltk qwue es un modelo de léxico.

Ten en cuenta que estos modelos preentrenados algunos califican en neutral. 

Por ejemplo,  `distilbert-base-uncased-finetuned-sst-2-english`. Es binario (POSITIVE / NEGATIVE, sin clase neutral) y trabaja con un máximo de 512 tokens, por lo que se usa `truncation=True`.

Muestra que % de los pros son positivas segun el modelo y quie % de cons son negativas segun en 

Muestra algún enjemplo de casos discordantes

In [ ]:
#'distilbert-base-uncased-finetuned-sst-2-english', ha tardado 37 min para 47000 reseñas

---
## 3. Topic modeling 

Podéis usar  BERTopic visto en clase o alguna otra técnica.



- **IMPORTANTE** **Un único modelo para las 5 empresas** (uno para pros y otro para cons). Si se entrenara un modelo por empresa, los topics no coincidirían y no se podrían comparar.
- Emplea etiquetas de negopcio para cada topic identificado. Por ejemplo: Compensación y beneficios,  Conciliación, Carrera y desarrollo, management , Cultura, 'Burocracia y procesos, ...
- Cuantso pros y cons salen por cada topic (%)
- ...



---
## 4. Comparación con peers

Para cada empresa calculo el % de reseñas cuyos pros caen en cada categoría, y lo mismo con los cons.

**Cuidado con la interpretación.** Los porcentajes de cada empresa suman 100%, así que son relativos: si una empresa tiene pocas quejas de salario, el peso de "salario" en sus cons baja, pero sube el del resto aunque nada haya empeorado. Por eso no basta con mirar los cons. Uso el **balance neto**:

> balance neto = % de reseñas que destacan la categoría en pros − % que la mencionan en cons

Un balance neto por encima del de los peers indica una fortaleza relativa; por debajo, una debilidad relativa.

---
## 5. Empleados actuales vs. antiguos

Los cons de quienes ya se han ido son, en la práctica, **motivos de salida**: el input más directo para un plan de retención.
Puedes comparar los topic de pros y cons de los empleados actuales vs empleados antiguos.
Puedes detectar motivos de abandono en los empleados antiguos. 

---
## 7. Conclusiones y recomendaciones para RRHH

- Iderntifica areas mejora y puntos fuertes de nuestra compañia
- identifica palancas para un posible plan de retención y para unb popsible plan de atraccion de nuevo talento.
